In [2]:
# -*- coding: utf-8 -*-
"""
Projeto Final - Computação Quântica
Algoritmo de Grover - Análise de Limiar de Ruído (Threshold Analysis)
Baseado em: Phys. Rev. A 102, 042609 (2020)
"""

%pip install qiskit qiskit_aer qiskit_ibm_runtime pylatexenc matplotlib numpy scipy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import math
import numpy as np
import json
import pickle
import warnings
import csv
from datetime import datetime
from collections import defaultdict
from scipy.interpolate import interp1d
from qiskit import QuantumCircuit
from qiskit.circuit.library import ZGate, MCMTGate
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error, pauli_error, ReadoutError
import matplotlib.pyplot as plt

# Suprimir warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Configurações Globais
SHOTS = 2048 # Aumentado para melhor resolução do P_hn
RUNS = 10
OPTIMIZATION_LEVEL = 1

print("="*80)
print("INICIALIZANDO EXPERIMENTO DE GROVER BASEADO EM WANG & KRSTIC (2020)")
print("="*80)

# =============================================================================
# FUNÇÃO DE MÉTRICA PRINCIPAL: SELETIVIDADE (SELECTIVITY)
# =============================================================================

def calculate_metrics(probs, marked_states, n_qubits, prob_ideal_ref=None):
    """
    Calcula métricas incluindo a Seletividade (S) definida no paper.

    Ref: "S = 10 log10(Pt / Phn)"
    Onde:
      - Pt: Probabilidade do estado alvo (Success Probability)
      - Phn: Probabilidade do maior sinal de ruído (Highest Noise)
    """
    N = 2 ** n_qubits
    M = len(marked_states)

    # 1. Probabilidade de Sucesso (Pt)
    success_prob = sum(probs.get(state, 0) for state in marked_states)

    # 2. Probabilidade do Maior Ruído (Phn) [cite: 36]
    # Encontrar a maior probabilidade entre os estados NÃO marcados
    max_noise_prob = 0.0
    all_possible_states = [format(i, f'0{n_qubits}b') for i in range(N)]

    for state in all_possible_states:
        if state not in marked_states:
            p = probs.get(state, 0)
            if p > max_noise_prob:
                max_noise_prob = p

    # 3. Cálculo da Seletividade (S)
    # Tratamento para evitar divisão por zero ou log de zero
    if max_noise_prob > 0 and success_prob > 0:
        ratio = success_prob / max_noise_prob
        selectivity = 10 * np.log10(ratio)
    elif max_noise_prob == 0 and success_prob > 0:
        selectivity = 100.0 # Valor alto arbitrário para "infinito" (separação perfeita)
    else:
        selectivity = -100.0 # Valor baixo para falha total

    # Outras métricas auxiliares
    if prob_ideal_ref is not None:
        deg_abs = prob_ideal_ref - success_prob
        deg_rel = (deg_abs / prob_ideal_ref * 100) if prob_ideal_ref > 0 else 0
    else:
        deg_abs = 0; deg_rel = 0

    # Fidelidade (Bhattacharyya distance squared)
    ideal_prob_per_marked = 1.0 / M if M > 0 else 0
    fidelity = 0
    for state in all_possible_states:
        p_ideal = ideal_prob_per_marked if state in marked_states else 0
        p_measured = probs.get(state, 0)
        fidelity += np.sqrt(p_ideal * p_measured)
    fidelity = fidelity ** 2

    return {
        'selectivity': selectivity,       # Métrica Principal
        'success_prob': success_prob,     # Pt [cite: 36]
        'max_noise_prob': max_noise_prob, # Phn [cite: 36]
        'fidelity': fidelity,
        'degradation_rel': deg_rel
    }

print("✓ Função calculate_metrics definida com Seletividade (Eq. 1)")